# Fusion Risk — End-to-End Walkthrough

This notebook drives the modular pipeline in `src/`. It reproduces the training,
explainability, and fairness audit shown in the deployed app.

**Prereq:** place `accepted_2007_to_2018Q4.csv` in `data/`, then run top to bottom.

In [ ]:
import sys; sys.path.insert(0, '..')
from src.data import build_subset, clean_and_engineer, temporal_split
from src import config as C

df = clean_and_engineer(build_subset())
train, test, kind = temporal_split(df)
print('split:', kind, '| train:', len(train), '| test:', len(test))
df[['int_rate','dti','fico_range_high','default']].describe()

## Train + compare all models (LR, RF, GB, XGBoost, LightGBM, CatBoost) with Optuna + MLflow

In [ ]:
from src.train import main
main(n_trials=25)   # writes models/best_model.joblib + metadata.json

## Explainability: SHAP + permutation + calibration

In [ ]:
import joblib
from src.explain import shap_global, permutation_importance_df, calibration_data

pipe = joblib.load(C.MODELS_DIR / 'best_model.joblib')
X, y = test.drop(columns=[C.TARGET]), test[C.TARGET]
print(shap_global(pipe, X.head(300)).head(10))
print(permutation_importance_df(pipe, X.head(1000), y.head(1000)).head(10))

## Fairness audit (Fairlearn, ECOA-oriented)

In [ ]:
from src.fairness import audit
table, summary = audit(pipe, X, y, test[C.SENSITIVE_COL])
display(table)
summary